<a href="https://colab.research.google.com/github/omkar-prabhu-github/Shipping_prediction/blob/main/shipping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install PySpark in the Colab cloud environment
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
import pandas as pd

# Start Spark
spark = SparkSession.builder.appName("ShippingDelayPrediction").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("PySpark installed and ready!")

PySpark installed and ready!


In [2]:
# Load Data
df = spark.read.csv("shipping.csv", header=True, inferSchema=True)

# Clean the data
safe_columns = [c.replace(".", "_") for c in df.columns]
df = df.toDF(*safe_columns)

df = df.drop("ID").na.drop()
df = df.withColumnRenamed("Reached_on_Time_Y_N", "label")

print("Data loaded and cleaned! First 5 rows:")
df.show(5)

Data loaded and cleaned! First 5 rows:
+---------------+----------------+-------------------+---------------+-------------------+---------------+------------------+------+----------------+-------------+-----+
|Warehouse_block|Mode_of_Shipment|Customer_care_calls|Customer_rating|Cost_of_the_Product|Prior_purchases|Product_importance|Gender|Discount_offered|Weight_in_gms|label|
+---------------+----------------+-------------------+---------------+-------------------+---------------+------------------+------+----------------+-------------+-----+
|              D|          Flight|                  4|              2|                177|              3|               low|     F|              44|         1233|    1|
|              F|          Flight|                  4|              5|                216|              2|               low|     M|              59|         3088|    1|
|              A|          Flight|                  2|              2|                183|              4|     

In [3]:
# Convert text columns to numbers
categorical_cols = ["Warehouse_block", "Mode_of_Shipment", "Product_importance", "Gender"]
indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx") for c in categorical_cols]

# Combine all features into one vector
numeric_cols = ["Customer_care_calls", "Customer_rating", "Cost_of_the_Product",
                "Prior_purchases", "Discount_offered", "Weight_in_gms"]
feature_cols = numeric_cols + [c+"_idx" for c in categorical_cols]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

print("Features ready!")

Features ready!


In [4]:
# Define Model and Pipeline
lr = LogisticRegression(featuresCol="features", labelCol="label")
pipeline = Pipeline(stages=indexers + [assembler, lr])

print("Training model... (This takes a few seconds)")
model = pipeline.fit(df)
print("Model trained successfully!")

Training model... (This takes a few seconds)
Model trained successfully!


In [5]:
# BEST CASE SCENARIO
new_data = spark.createDataFrame([
    ("F", "Flight", "high", "M", 2, 5, 150, 5, 65, 1200)
], ["Warehouse_block", "Mode_of_Shipment", "Product_importance", "Gender",
    "Customer_care_calls", "Customer_rating", "Cost_of_the_Product",
    "Prior_purchases", "Discount_offered", "Weight_in_gms"])

# Predict
result = model.transform(new_data)
final_output = result.select("prediction")

print("Prediction: 1.0 = On Time | 0.0 = Delayed\n")

display(final_output.toPandas())

Prediction: 1.0 = On Time | 0.0 = Delayed



,prediction
0,1.0


In [6]:
# Worst Case Scenario
new_data = spark.createDataFrame([
    ("A", "Ship", "low", "M", 7, 1, 300, 2, 1, 7500)
], ["Warehouse_block", "Mode_of_Shipment", "Product_importance", "Gender",
    "Customer_care_calls", "Customer_rating", "Cost_of_the_Product",
    "Prior_purchases", "Discount_offered", "Weight_in_gms"])

# Run the prediction
result = model.transform(new_data)
final_output = result.select("prediction")

print("Prediction: 1.0 = On Time | 0.0 = Delayed\n")

display(final_output.toPandas())

Prediction: 1.0 = On Time | 0.0 = Delayed



,prediction
0,0.0
